# Orbit Studio Cloud Engine

*Free GPU processing for your 360 captures.*

## What this notebook does

Run the cells in order, top to bottom. Each one builds on the last, and none of them need Colab Pro, a paid plan, or anything beyond the free T4 GPU Colab already gives you.

- [ ] Confirm the runtime has a GPU
- [ ] Upload the capture bundle from Orbit Studio
- [ ] Install the reconstruction and training tools
- [ ] Recover camera poses with COLMAP
- [ ] Train a Gaussian splat with gsplat
- [ ] Export the splat and download it back to your laptop

Two optional cells sit at the end for anyone who wants to experiment further.

## 1. Confirm you have a GPU

Colab hands out GPUs for free, but only if you ask for one. This cell checks for an NVIDIA GPU and stops early with clear instructions if none is attached, so you find out now rather than forty minutes into training.

In [ ]:
import subprocess

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    if result.returncode != 0 or "no devices" in result.stdout.lower():
        raise RuntimeError("nvidia-smi ran but reported no GPU")
    print(result.stdout)
    print("GPU detected. You are ready to continue.")
except Exception as error:
    print(f"No GPU is attached to this runtime: {error}")
    print("Open the Runtime menu, choose Change runtime type, pick T4 GPU under Hardware accelerator, then Save.")
    print("Then run this cell again.")
    raise


## 2. Upload your capture bundle

Orbit Studio packages your extracted frames and a manifest into a single `bundle.zip` on your laptop. Upload that file here. It gets unzipped into `/content/work/images`, and if a `manifest.json` is present, this cell prints a short summary of what you shot.

In [ ]:
import os
import json
import shutil
import zipfile
from google.colab import files

work_dir = "/content/work"
images_dir = os.path.join(work_dir, "images")
os.makedirs(work_dir, exist_ok=True)

try:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded")

    zip_name = list(uploaded.keys())[0]
    zip_path = os.path.join(work_dir, zip_name)
    with open(zip_path, "wb") as f:
        f.write(uploaded[zip_name])

    if os.path.exists(images_dir):
        shutil.rmtree(images_dir)
    os.makedirs(images_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(images_dir)

    entries = os.listdir(images_dir)
    has_images_here = any(name.lower().endswith((".jpg", ".jpeg", ".png")) for name in entries)
    nested_dirs = [name for name in entries if os.path.isdir(os.path.join(images_dir, name))]
    if not has_images_here and len(nested_dirs) == 1:
        nested_path = os.path.join(images_dir, nested_dirs[0])
        for item in os.listdir(nested_path):
            shutil.move(os.path.join(nested_path, item), images_dir)
        os.rmdir(nested_path)

    stray_manifest = os.path.join(images_dir, "manifest.json")
    manifest_path = os.path.join(work_dir, "manifest.json")
    if os.path.exists(stray_manifest):
        shutil.move(stray_manifest, manifest_path)

    image_files = [name for name in os.listdir(images_dir) if name.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(f"Extracted {len(image_files)} images to {images_dir}")

    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        print("Capture summary:")
        for key in ("project", "crops", "rig"):
            if key in manifest:
                print(f"  {key}: {manifest[key]}")
    else:
        print("No manifest.json found. Continuing with images only, that is fine.")

except zipfile.BadZipFile:
    print(f"{zip_name} does not look like a valid zip file.")
    print("Re-export the bundle from Orbit Studio and upload it again.")
    raise
except Exception as error:
    print(f"Upload step failed: {error}")
    print("Run this cell again and choose the bundle.zip file exported by Orbit Studio.")
    raise


### Working with a larger capture

If your bundle is too large for a browser upload, put it in Google Drive instead and mount your Drive before running the cell above:

```
from google.colab import drive
drive.mount('/content/drive')
```

Then copy or unzip your bundle from the mounted Drive path into `/content/work/images` before continuing to the install step.

## 3. Install the toolchain

This installs pycolmap for camera pose recovery, then clones and installs gsplat for training. Long installs run quietly and print a done line when finished, and version numbers are printed at the end so you have a record of exactly what ran.

In [ ]:
import os
import subprocess
import time

def run(command, label):
    print(f"Installing {label} ...")
    start = time.time()
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    elapsed = time.time() - start
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"{label} failed after {elapsed:.0f}s")
    print(f"{label} done in {elapsed:.0f}s")

try:
    run("pip install -q pycolmap numpy pillow", "pycolmap, numpy, pillow")

    if not os.path.exists("gsplat"):
        run("git clone --quiet --depth 1 https://github.com/nerfstudio-project/gsplat.git", "gsplat clone")
    else:
        print("gsplat already cloned, skipping")

    run("pip install -q ./gsplat", "gsplat package")
    run("pip install -q -r gsplat/examples/requirements.txt", "gsplat example requirements")

    import pycolmap
    import numpy
    import PIL
    import torch

    print(f"pycolmap {pycolmap.__version__}")
    print(f"numpy {numpy.__version__}")
    print(f"pillow {PIL.__version__}")
    print(f"torch {torch.__version__}")
    print("All installs finished. Continue to the next cell.")

except Exception as error:
    print(f"Install step failed: {error}")
    print("Try running this cell again. If it keeps failing, restart the runtime from the Runtime menu and start over from the GPU check cell.")
    raise


## 4. Recover camera poses

Splat training needs to know where each frame was taken from. This cell runs COLMAP's feature extraction and matching, then incremental mapping to solve for camera positions.

It uses a per-image camera model rather than one shared camera. Orbit Studio's extracted frames are not all identical crops, and forcing a single shared camera model onto mixed crops produces bad poses, so per-image is the safer default here.

If fewer than 60 percent of your images register, the cell warns you and points at the capture guide. That number is a real signal the walk needs more overlap, not a bug to shrug off.

In [ ]:
import os
import shutil
import pycolmap

work_dir = "/content/work"
images_dir = os.path.join(work_dir, "images")
database_path = os.path.join(work_dir, "database.db")
sparse_dir = os.path.join(work_dir, "sparse")

try:
    os.makedirs(sparse_dir, exist_ok=True)
    if os.path.exists(database_path):
        os.remove(database_path)

    total_images = len([name for name in os.listdir(images_dir) if name.lower().endswith((".jpg", ".jpeg", ".png"))])
    if total_images < 10:
        raise RuntimeError(f"Only {total_images} images found, that is not enough overlap to reconstruct a scene")

    print(f"Extracting features from {total_images} images ...")
    # camera_model is NOT a parameter of extract_features - it lives on
    # ImageReaderOptions. Passing it directly raises TypeError before a single
    # feature is extracted, which is what this cell used to do every time.
    reader_options = pycolmap.ImageReaderOptions()
    reader_options.camera_model = "OPENCV"
    pycolmap.extract_features(
        database_path=database_path,
        image_path=images_dir,
        camera_mode=pycolmap.CameraMode.PER_IMAGE,
        reader_options=reader_options,
    )
    print("Feature extraction complete.")

    print("Matching images ...")
    if hasattr(pycolmap, "match_sequential"):
        pycolmap.match_sequential(database_path=database_path)
        print("Used sequential matching.")
    else:
        pycolmap.match_exhaustive(database_path=database_path)
        print("Used exhaustive matching.")

    print("Running incremental mapping, this can take a while on a large capture ...")
    reconstructions = pycolmap.incremental_mapping(
        database_path=database_path,
        image_path=images_dir,
        output_path=sparse_dir,
    )

    if not reconstructions:
        raise RuntimeError("Incremental mapping produced no reconstruction, the scene likely has too little overlap between frames")

    best_key = max(reconstructions, key=lambda key: reconstructions[key].num_reg_images())
    best_model = reconstructions[best_key]
    registered = best_model.num_reg_images()
    ratio = registered / total_images if total_images else 0

    print(f"Registered {registered} of {total_images} images, {ratio * 100:.1f} percent.")

    final_sparse_dir = os.path.join(sparse_dir, "0")
    if str(best_key) != "0":
        source_dir = os.path.join(sparse_dir, str(best_key))
        if os.path.exists(final_sparse_dir):
            shutil.rmtree(final_sparse_dir)
        shutil.move(source_dir, final_sparse_dir)

    if ratio < 0.6:
        print("Warning: fewer than 60 percent of images registered.")
        print("This usually means gaps in coverage, motion blur, or too little overlap between frames.")
        print("Check docs/CAPTURE_GUIDE.md on your laptop for the walking pattern, then re-shoot the weak area.")
    else:
        print("Registration looks healthy. Continue to training.")

except Exception as error:
    print(f"Pose estimation failed: {error}")
    print("Confirm the upload cell extracted images correctly, and that frames have real visual overlap between them.")
    raise


## 5. Train the splat

This runs gsplat's example trainer against the poses from the last cell. Expect 30 to 60 minutes on a free T4 for a room-sized capture. Colab may dim the tab or ask if you are still there while this runs. You can do other things in the meantime, just do not close the tab or let the runtime disconnect.

In [ ]:
import os
import subprocess
import sys
import time

script_path = "gsplat/examples/simple_trainer.py"

# Checkpoints go to Drive when it will mount. Training is 30 to 60 minutes and the
# Colab VM is wiped the instant it disconnects, so keeping the only copy under
# /content means an idle tab can cost the entire run - which is exactly what
# happened on the first real capture. Drive outlives the runtime.
result_dir = "/content/work/results"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    result_dir = "/content/drive/MyDrive/orbit-studio/results"
    os.makedirs(result_dir, exist_ok=True)
    print(f"Checkpoints will be written to your Drive: {result_dir}")
except Exception as mount_error:
    print(f"Drive not mounted: {mount_error}")
    print(f"Checkpoints stay on the runtime at {result_dir}, and a disconnect LOSES them.")
    print("Re-run this cell and accept the Drive prompt if you would rather not risk it.")

try:
    help_result = subprocess.run([sys.executable, script_path, "default", "--help"], capture_output=True, text=True)
    help_text = help_result.stdout + help_result.stderr

    command = [
        sys.executable, script_path, "default",
        "--data_dir", "/content/work",
        "--data_factor", "1",
        "--result_dir", result_dir,
        "--max_steps", "15000",
    ]

    if "--disable_viewer" in help_text:
        command.append("--disable_viewer")
        print("Viewer disabled for headless training.")

    print("Training started. Expect 30 to 60 minutes on a free T4.")
    start = time.time()
    result = subprocess.run(command, text=True)
    elapsed = time.time() - start

    if result.returncode != 0:
        raise RuntimeError(f"Training exited with code {result.returncode} after {elapsed / 60:.1f} minutes")

    print(f"Training finished in {elapsed / 60:.1f} minutes.")

except FileNotFoundError:
    print(f"Could not find {script_path}")
    print("Re-run the install cell to make sure gsplat cloned correctly.")
    raise
except Exception as error:
    print(f"Training failed: {error}")
    print("If this looks like an out-of-memory error, lower --max_steps in this cell or confirm no other training run is active.")
    raise


## 6. Export your splat

This finds the newest checkpoint gsplat produced, loads it, and writes two files. `artifact.ply` is a standard 3D Gaussian Splat file that many tools can read. `artifact.splat` is the compact antimatter15 format Orbit Studio's viewer expects. Both downloads start automatically. Drop `artifact.splat` into Orbit Studio's Import Result zone to fly through your scene.

In [ ]:
import glob
import os
import numpy as np
import torch

# Drive first, then the runtime. Training writes to Drive when it mounted, and a
# checkpoint there survives the VM - so a disconnect costs you this export step,
# never the 40 minutes behind it. Newest wins when both exist.
RESULT_DIRS = [
    "/content/drive/MyDrive/orbit-studio/results",
    "/content/work/results",
]
OUTPUT_DIR = "/content/drive/MyDrive/orbit-studio" if os.path.isdir("/content/drive/MyDrive") else "/content/work"
os.makedirs(OUTPUT_DIR, exist_ok=True)
ply_path = os.path.join(OUTPUT_DIR, "artifact.ply")
splat_path = os.path.join(OUTPUT_DIR, "artifact.splat")

def find_latest_checkpoint(base_dirs):
    candidates = []
    for base_dir in base_dirs:
        candidates.extend(glob.glob(os.path.join(base_dir, "**", "*.pt"), recursive=True))
    if not candidates:
        raise FileNotFoundError(f"No .pt checkpoint files found under any of: {base_dirs}")
    candidates.sort(key=os.path.getmtime)
    return candidates[-1]

def dig(node, path):
    for part in path.split("."):
        if isinstance(node, dict) and part in node:
            node = node[part]
        else:
            return None
    return node

def find_field(checkpoint, source, names):
    for name in names:
        if "." in name:
            value = dig(checkpoint, name)
        else:
            value = source.get(name) if isinstance(source, dict) else None
        if value is not None:
            return value
    return None

def to_numpy(tensor):
    if hasattr(tensor, "detach"):
        return tensor.detach().cpu().numpy()
    return np.asarray(tensor)

try:
    checkpoint_path = find_latest_checkpoint(RESULT_DIRS)
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    source = checkpoint["splats"] if isinstance(checkpoint, dict) and isinstance(checkpoint.get("splats"), dict) else checkpoint

    means = find_field(checkpoint, source, ["means", "splats.means", "means3d"])
    quats = find_field(checkpoint, source, ["quats", "splats.quats", "rotation"])
    scales = find_field(checkpoint, source, ["scales", "splats.scales", "scaling"])
    opacities = find_field(checkpoint, source, ["opacities", "splats.opacities", "opacity"])
    sh0 = find_field(checkpoint, source, ["sh0", "splats.sh0", "features_dc", "shs_0"])

    fields = {"means": means, "quats": quats, "scales": scales, "opacities": opacities, "sh0": sh0}
    missing = [name for name, value in fields.items() if value is None]
    if missing:
        raise KeyError(f"Checkpoint is missing expected fields: {missing}, the gsplat layout may have changed")

    means = to_numpy(means).astype(np.float32).reshape(-1, 3)
    quats = to_numpy(quats).astype(np.float32).reshape(-1, 4)
    scales = to_numpy(scales).astype(np.float32).reshape(-1, 3)
    opacities = to_numpy(opacities).astype(np.float32).reshape(-1)
    sh0 = to_numpy(sh0).astype(np.float32).reshape(-1, 3)

    count = means.shape[0]
    print(f"Loaded {count} splats from checkpoint.")

    header_lines = [
        "ply",
        "format binary_little_endian 1.0",
        f"element vertex {count}",
        "property float x",
        "property float y",
        "property float z",
        "property float nx",
        "property float ny",
        "property float nz",
        "property float f_dc_0",
        "property float f_dc_1",
        "property float f_dc_2",
        "property float opacity",
        "property float scale_0",
        "property float scale_1",
        "property float scale_2",
        "property float rot_0",
        "property float rot_1",
        "property float rot_2",
        "property float rot_3",
        "end_header",
    ]

    normals = np.zeros((count, 3), dtype=np.float32)
    ply_records = np.concatenate([means, normals, sh0, opacities.reshape(-1, 1), scales, quats], axis=1).astype(np.float32)

    with open(ply_path, "wb") as f:
        f.write(("\n".join(header_lines) + "\n").encode("ascii"))
        f.write(ply_records.tobytes())
    print(f"Wrote {ply_path}")

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    sh_c0 = 0.2820947917738781
    linear_scales = np.exp(scales)
    rgb = np.clip((0.5 + sh_c0 * sh0) * 255.0, 0, 255).astype(np.uint8)
    alpha = np.clip(sigmoid(opacities) * 255.0, 0, 255).astype(np.uint8).reshape(-1, 1)
    rgba = np.concatenate([rgb, alpha], axis=1).astype(np.uint8)

    norm = np.linalg.norm(quats, axis=1, keepdims=True)
    norm[norm == 0] = 1.0
    rot_bytes = np.clip((quats / norm) * 128.0 + 128.0, 0, 255).astype(np.uint8)

    splat_dtype = np.dtype([("pos", "<f4", 3), ("scale", "<f4", 3), ("rgba", "u1", 4), ("rot", "u1", 4)])
    splat_array = np.zeros(count, dtype=splat_dtype)
    splat_array["pos"] = means
    splat_array["scale"] = linear_scales
    splat_array["rgba"] = rgba
    splat_array["rot"] = rot_bytes

    with open(splat_path, "wb") as f:
        f.write(splat_array.tobytes())
    print(f"Wrote {splat_path}")

    # Only artifact.splat downloads automatically. Two back-to-back files.download()
    # calls trip Chrome's "site wants to download multiple files" block, which kills
    # the SECOND one silently - and a browser that restricts downloads at all, as a
    # managed work machine usually does, quietly drops both. files.download() does
    # not raise when the browser ignores it, so the cell would happily print
    # "Downloads started" while nothing ever arrived. artifact.ply stays on disk and
    # is named below rather than pushed through the same fragile path twice.
    print(f"artifact.ply is on the runtime at {ply_path} if you want it too.")
    try:
        from google.colab import files
        files.download(splat_path)
        print("Download started for artifact.splat.")
    except Exception as download_error:
        print(f"Automatic download did not start: {download_error}")

    print()
    print("If artifact.splat is not in your Downloads folder, the browser blocked it.")
    print("The file is safe on the runtime - get it either way:")
    print("  - Files panel: the folder icon on the left, open 'work', right-click")
    print("    artifact.splat, Download.")
    print("  - Or copy it to your Drive, which no download policy can block:")
    print("      from google.colab import drive; drive.mount('/content/drive')")
    print(f"      import shutil; shutil.copy('{splat_path}', '/content/drive/MyDrive/artifact.splat')")
    print()
    print("Do it before the runtime disconnects - the VM is wiped when it does.")

except Exception as error:
    print(f"Export failed: {error}")
    print(f"Looked for a checkpoint under: {RESULT_DIRS}")
    print("If Drive was mounted during training, remount it and run this cell again.")
    raise


## Optional: experimental feed-forward lane

Everything above is the reliable path. AnySplat is a newer feed-forward model that can produce a splat straight from a handful of images, skipping COLMAP and the training loop entirely, in exchange for lower fidelity and a rougher setup experience. It is included here for anyone who wants to try it, and it is entirely optional. Skip this if you already have a good `artifact.splat` from the cells above.

In [ ]:
import os
import subprocess

try:
    print("This lane is experimental and may need version pinning against a specific commit.")
    if not os.path.exists("AnySplat"):
        subprocess.run("git clone --quiet https://github.com/InternRobotics/AnySplat", shell=True, check=True)
    print("AnySplat cloned. Follow its README to run feed-forward inference on a subset of /content/work/images.")
    print("This cell stops short of auto-running inference on purpose, AnySplat's interface changes often enough that a fixed call here would go stale.")
except Exception as error:
    print(f"AnySplat setup did not complete: {error}")
    print("This lane is optional. Your artifact.splat from the gsplat lane above is unaffected.")


## Optional: single-panorama lane

If you only have one equirectangular photo rather than a full walk-through, the pipeline above is more than you need. SPAG4d (MIT license, github.com/cedarconnor/SPAG4d) turns a single equirectangular photo into a splat in seconds, and ships a portable Windows build for NVIDIA machines. It runs entirely on your own PC, outside this notebook.